In [1]:
# =============================================================================
# HUMOB / SIGSPATIAL Cup 2025 - MEMORY-EFFICIENT FINAL IMPLEMENTATION
# =============================================================================

import subprocess
import sys
print("Installing requirements...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", 
                      "git+https://github.com/yahoojapan/geobleu.git", "tqdm", "scikit-learn", "pandas", "numpy"])

import os
import gc
import time
import pandas as pd
import numpy as np
from collections import defaultdict, Counter
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

try:
    from geobleu import calc_geobleu_bulk
    print("GeoBLEU imported successfully.\n")
except Exception as e:
    print(f"Failed to import geobleu: {e}")
    calc_geobleu_bulk = None

# =============================================================================
# CONFIGURATION
# =============================================================================
DATA_DIR = "/kaggle/input/humob-data/15313913"
CITIES = ["B", "C", "D"]  # Run one city at a time
COLUMNS = ["uid","d","t","x","y"]
DTYPES = {"uid":"int32","d":"int16","t":"int16","x":"int16","y":"int16"}

TRAIN_DAY_MAX = 60
TEST_DAY_MIN = 61
MASK_VALUE = 999
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

TARGET_RANGES = {
    "A":(147001,150000), "B":(27001,30000), "C":(22001,25000), "D":(17001,20000)
}

# Hyperparameters - optimized for memory
TOP_K = 8
VAL_FRACTION = 0.2  # Smaller validation set to save memory
POI_TOP_N = 25
PADDING = -999

OUT_DIR = "./results"
os.makedirs(OUT_DIR, exist_ok=True)

# =============================================================================
# UTILITIES
# =============================================================================

def load_city_df(city):
    """Load city data."""
    path = os.path.join(DATA_DIR, f"city_{city}_challengedata.csv")
    if not os.path.exists(path):
        path = os.path.join(DATA_DIR, f"city_{city.lower()}_challengedata.csv")
    print(f"Loading: {path}")
    return pd.read_csv(path, usecols=COLUMNS, dtype=DTYPES)

def to_loc(x, y, max_y):
    """(x,y) -> loc_id"""
    return int(x * (max_y + 1) + y)

def from_loc(loc, max_y):
    """loc_id -> (x,y)"""
    if max_y <= 0:
        return (0, 0)
    return int(loc // (max_y + 1)), int(loc % (max_y + 1))

# =============================================================================
# MEMORY-EFFICIENT PROFILING
# =============================================================================

def build_profiles_efficient(df, max_y):
    """
    Build user profiles efficiently using numpy arrays.
    Stores only essential data in compact format.
    """
    print(f"Building profiles for {df['uid'].nunique()} users...")
    profiles = {}
    
    # Group by user
    for uid, user_df in tqdm(df.groupby('uid'), desc="Profiling"):
        uid = int(uid)
        
        # Store day signatures as dict of dicts (memory efficient)
        day_sigs = defaultdict(dict)
        loc_counter = Counter()
        hourly = defaultdict(Counter)
        
        for row in user_df.itertuples():
            loc = to_loc(row.x, row.y, max_y)
            day_sigs[row.d][row.t] = loc
            loc_counter[loc] += 1
            hourly[row.t][loc] += 1
        
        # Get fallback location
        fallback = from_loc(loc_counter.most_common(1)[0][0], max_y) if loc_counter else (0, 0)
        
        # Store compact profile
        profiles[uid] = {
            'days': dict(day_sigs),  # Convert defaultdict to dict to save memory
            'fallback': fallback,
            'hourly': {t: dict(c) for t, c in hourly.items()}  # Convert to dict
        }
        
        # Clear temporary structures
        del day_sigs, loc_counter, hourly
    
    gc.collect()
    print(f"Profiles built: {len(profiles)}")
    return profiles

def compute_pois(df, max_y, top_n):
    """Compute POIs per timestamp."""
    print(f"Computing top {top_n} POIs...")
    pois = {}
    
    for t in range(24):
        t_df = df[df['t'] == t]
        if not t_df.empty:
            locs = [to_loc(row.x, row.y, max_y) for row in t_df.itertuples()]
            pois[t] = set([loc for loc, _ in Counter(locs).most_common(top_n)])
    
    print("POI computation done")
    return pois

# =============================================================================
# STRATEGIES
# =============================================================================

def get_candidates(profile, clue, score_func, k=TOP_K):
    """Get top-k candidate days based on scoring."""
    if not clue:
        return []
    
    scores = {}
    for day, sig in profile['days'].items():
        s = score_func(sig, clue)
        if s > 0:
            scores[day] = s
    
    if not scores:
        return []
    
    top = sorted(scores, key=scores.get, reverse=True)[:k]
    return [profile['days'][d] for d in top]

def vote(slots, candidates, profile, max_y, clue=None):
    """Vote across candidates with fallback."""
    preds = []
    fallback_loc = to_loc(profile['fallback'][0], profile['fallback'][1], max_y)
    
    for t in slots:
        votes = [c[t] for c in candidates if t in c]
        
        if votes:
            counts = Counter(votes)
            loc = counts.most_common(1)[0][0]
        elif t in profile['hourly'] and profile['hourly'][t]:
            # Use hourly fallback
            loc = max(profile['hourly'][t], key=profile['hourly'][t].get)
        else:
            loc = fallback_loc
        
        preds.append((t, loc))
    
    return preds

# Strategy 1: Exact Match PPM
def strategy_exact(uid, clue, slots, profiles, max_y, **kw):
    """Exact timestamp matching."""
    profile = profiles.get(uid, {'days': {}, 'fallback': (0, 0), 'hourly': {}})
    
    def score(sig, clue):
        return sum(1 for t, loc in clue.items() if sig.get(t) == loc)
    
    cands = get_candidates(profile, clue, score)
    return vote(slots, cands, profile, max_y)

# Strategy 2: Fuzzy Time Matching
def strategy_fuzzy(uid, clue, slots, profiles, max_y, **kw):
    """Fuzzy matching with time windows."""
    profile = profiles.get(uid, {'days': {}, 'fallback': (0, 0), 'hourly': {}})
    
    def score(sig, clue):
        s = 0
        for t, loc in clue.items():
            if sig.get(t) == loc:
                s += 2  # Exact match
            else:
                # Check neighbors
                for dt in [-1, 1]:
                    if sig.get(t + dt) == loc:
                        s += 0.5
                        break
        return s
    
    cands = get_candidates(profile, clue, score)
    return vote(slots, cands, profile, max_y)

# Strategy 3: Spatial Distance
def strategy_spatial(uid, clue, slots, profiles, max_y, pois=None, **kw):
    """Spatial distance with POI boost."""
    profile = profiles.get(uid, {'days': {}, 'fallback': (0, 0), 'hourly': {}})
    
    clue_times = sorted(clue.keys())
    if not clue_times:
        cands = []
    else:
        # Build clue vector
        clue_vec = []
        weights = []
        for t in clue_times:
            x, y = from_loc(clue[t], max_y)
            clue_vec.extend([x, y])
            # POI weighting
            w = 2.0 if pois and clue[t] in pois.get(t, set()) else 1.0
            weights.extend([w, w])
        
        clue_vec = np.array(clue_vec, dtype=np.float32)
        weights = np.array(weights, dtype=np.float32)
        
        def score(sig, clue):
            day_vec = []
            for t in clue_times:
                if t in sig:
                    x, y = from_loc(sig[t], max_y)
                    day_vec.extend([x, y])
                else:
                    day_vec.extend([PADDING, PADDING])
            
            day_vec = np.array(day_vec, dtype=np.float32)
            dist = np.linalg.norm((clue_vec - day_vec) * weights)
            return 1.0 / (1.0 + dist)
        
        cands = get_candidates(profile, clue, score)
    
    return vote(slots, cands, profile, max_y)

# Strategy 4: Frequency-Based
def strategy_frequency(uid, clue, slots, profiles, max_y, **kw):
    """Use hourly frequency patterns."""
    profile = profiles.get(uid, {'days': {}, 'fallback': (0, 0), 'hourly': {}})
    
    # For this strategy, directly use hourly patterns
    preds = []
    fallback = to_loc(profile['fallback'][0], profile['fallback'][1], max_y)
    
    for t in slots:
        if t in profile['hourly'] and profile['hourly'][t]:
            loc = max(profile['hourly'][t], key=profile['hourly'][t].get)
        else:
            loc = fallback
        preds.append((t, loc))
    
    return preds

# =============================================================================
# VALIDATION (USER-BASED SPLIT)
# =============================================================================

def validate_strategy(val_df, profiles, max_y, strategy, **kwargs):
    """
    Proper validation:
    - Use revealed timestamps as clues
    - Predict only masked timestamps
    - Compare predictions to ground truth
    """
    preds = []
    gts = []
    
    for (uid, day), group in tqdm(val_df.groupby(['uid', 'd']), desc=f"Val {strategy.__name__}"):
        uid = int(uid)
        
        # Build clue from revealed data
        clue = {}
        for row in group.itertuples():
            if row.x != MASK_VALUE:
                clue[row.t] = to_loc(row.x, row.y, max_y)
        
        # Identify slots to predict (those that are masked)
        masked_slots = []
        gt_map = {}
        for row in group.itertuples():
            if row.x == MASK_VALUE:
                masked_slots.append(row.t)
                # In real validation, we need ground truth
                # For now, we'll skip truly masked data
        
        # For validation, we artificially mask some data
        # Let's predict all slots and compare to known ground truth
        all_gt = {row.t: (row.x, row.y) for row in group.itertuples() if row.x != MASK_VALUE}
        
        if len(all_gt) < 2:
            continue
        
        # Artificially create validation scenario:
        # Use first 50% of timestamps as clue, predict rest
        sorted_times = sorted(all_gt.keys())
        split_point = len(sorted_times) // 2
        
        clue = {t: to_loc(all_gt[t][0], all_gt[t][1], max_y) for t in sorted_times[:split_point]}
        test_slots = sorted_times[split_point:]
        
        if not test_slots:
            continue
        
        # Predict
        pred_locs = strategy(uid, clue, test_slots, profiles, max_y, **kwargs)
        
        # Compare
        for t, pred_loc in pred_locs:
            if t in all_gt:
                pred_x, pred_y = from_loc(pred_loc, max_y)
                gt_x, gt_y = all_gt[t]
                preds.append((uid, day, t, pred_x, pred_y))
                gts.append((uid, day, t, gt_x, gt_y))
    
    return preds, gts

def ensemble(pred_map, gts):
    """Weighted ensemble."""
    if not gts or not pred_map:
        return 0.0
    
    weights = {
        'strategy_exact': 2.5,
        'strategy_fuzzy': 2.0,
        'strategy_spatial': 1.5,
        'strategy_frequency': 1.0
    }
    
    votes = defaultdict(Counter)
    gt_map = {(u, d, t): (x, y) for u, d, t, x, y in gts}
    
    for name, preds in pred_map.items():
        w = weights.get(name, 1)
        for u, d, t, x, y in preds:
            if (u, d, t) in gt_map:
                votes[(u, d, t)][(x, y)] += w
    
    final_preds = []
    final_gts = []
    for key, counter in votes.items():
        if counter:
            (x, y), _ = counter.most_common(1)[0]
            final_preds.append((key[0], key[1], key[2], x, y))
            final_gts.append((key[0], key[1], key[2], gt_map[key][0], gt_map[key][1]))
    
    if not final_preds:
        return 0.0
    
    return calc_geobleu_bulk(final_preds, final_gts, processes=1)

# =============================================================================
# SUBMISSION GENERATION
# =============================================================================

def generate_submission(test_df, profiles, max_y, pois, city):
    """Generate submission efficiently."""
    print("\nGenerating submission predictions...")
    
    all_preds = {
        'exact': [],
        'fuzzy': [],
        'spatial': [],
        'freq': []
    }
    
    strategies = [
        ('exact', strategy_exact),
        ('fuzzy', strategy_fuzzy),
        ('spatial', strategy_spatial),
        ('freq', strategy_frequency)
    ]
    
    for name, strategy in strategies:
        print(f"Running {name}...")
        for (uid, day), group in tqdm(test_df.groupby(['uid', 'd']), desc=name):
            uid = int(uid)
            
            clue = {}
            masked = []
            
            for row in group.itertuples():
                if row.x != MASK_VALUE:
                    clue[row.t] = to_loc(row.x, row.y, max_y)
                else:
                    masked.append(row.t)
            
            if masked:
                preds = strategy(uid, clue, masked, profiles, max_y, pois=pois)
                for t, loc in preds:
                    x, y = from_loc(loc, max_y)
                    all_preds[name].append((uid, day, t, x, y))
    
    # Ensemble
    print("Ensembling...")
    weights = {'exact': 2.5, 'fuzzy': 2.0, 'spatial': 1.5, 'freq': 1.0}
    votes = defaultdict(Counter)
    
    for name, preds in all_preds.items():
        w = weights[name]
        for u, d, t, x, y in preds:
            votes[(u, d, t)][(x, y)] += w
    
    rows = []
    for (uid, d, t), counter in votes.items():
        (x, y), _ = counter.most_common(1)[0]
        rows.append({'uid': uid, 'd': d, 't': t, 'x': x, 'y': y})
    
    sub_df = pd.DataFrame(rows)
    
    # Filter to target users
    lo, hi = TARGET_RANGES[city]
    sub_df = sub_df[sub_df['uid'].between(lo, hi)]
    
    out_file = os.path.join(OUT_DIR, f"submission_{city}.csv")
    sub_df.to_csv(out_file, index=False)
    print(f"Saved: {out_file} ({len(sub_df)} predictions)")
    
    return sub_df

# =============================================================================
# MAIN
# =============================================================================

def main():
    start = time.time()
    
    for city in CITIES:
        print("\n" + "="*60)
        print(f"CITY {city}")
        print("="*60)
        
        # Load data
        df = load_city_df(city)
        labeled = df[df['x'] != MASK_VALUE]
        
        if labeled.empty:
            print("No labeled data!")
            continue
        
        MAX_Y = int(labeled['y'].max())
        print(f"Grid: {int(labeled['x'].max())} x {MAX_Y}")
        
        # USER-BASED SPLIT for validation
        print(f"\nSplitting users for validation...")
        future_labeled = df[(df['d'] >= TEST_DAY_MIN) & (df['x'] != MASK_VALUE)]
        future_users = future_labeled['uid'].unique()
        
        if len(future_users) == 0:
            print("No validation users!")
            continue
        
        # Split users (not rows!)
        train_users, val_users = train_test_split(
            future_users, 
            test_size=VAL_FRACTION, 
            random_state=RANDOM_SEED
        )
        
        print(f"Train users: {len(train_users)}, Val users: {len(val_users)}")
        
        # Build profiles from training users only
        train_data = pd.concat([
            df[(df['d'] <= TRAIN_DAY_MAX) & (df['x'] != MASK_VALUE)],
            future_labeled[future_labeled['uid'].isin(train_users)]
        ])
        
        profiles = build_profiles_efficient(train_data, MAX_Y)
        pois = compute_pois(train_data, MAX_Y, POI_TOP_N)
        
        del train_data
        gc.collect()
        
        # Validation on held-out users
        val_data = future_labeled[future_labeled['uid'].isin(val_users)]
        
        if val_data.empty or calc_geobleu_bulk is None:
            print("No validation data!")
            continue
        
        print("\n" + "-"*60)
        print("VALIDATION (on held-out users)")
        print("-"*60)
        
        kwargs = {'profiles': profiles, 'max_y': MAX_Y, 'pois': pois}
        
        p1, g1 = validate_strategy(val_data, strategy=strategy_exact, **kwargs)
        s1 = calc_geobleu_bulk(p1, g1, processes=1) if p1 else 0.0
        print(f"Strategy 1 (Exact): {s1:.6f}")
        
        p2, g2 = validate_strategy(val_data, strategy=strategy_fuzzy, **kwargs)
        s2 = calc_geobleu_bulk(p2, g2, processes=1) if p2 else 0.0
        print(f"Strategy 2 (Fuzzy): {s2:.6f}")
        
        p3, g3 = validate_strategy(val_data, strategy=strategy_spatial, **kwargs)
        s3 = calc_geobleu_bulk(p3, g3, processes=1) if p3 else 0.0
        print(f"Strategy 3 (Spatial): {s3:.6f}")
        
        p4, g4 = validate_strategy(val_data, strategy=strategy_frequency, **kwargs)
        s4 = calc_geobleu_bulk(p4, g4, processes=1) if p4 else 0.0
        print(f"Strategy 4 (Frequency): {s4:.6f}")
        
        pred_map = {
            'strategy_exact': p1,
            'strategy_fuzzy': p2,
            'strategy_spatial': p3,
            'strategy_frequency': p4
        }
        
        ens_score = ensemble(pred_map, g1)
        
        print("-"*60)
        print(f"ENSEMBLE: {ens_score:.6f}")
        print("="*60)
        
        # Clean up validation data
        del val_data, p1, p2, p3, p4, g1, g2, g3, g4, pred_map
        gc.collect()
        
        # Generate submission (reuse existing profiles!)
        test_data = df[df['d'] >= TEST_DAY_MIN].copy()
        generate_submission(test_data, profiles, MAX_Y, pois, city)
        
        # Clean up
        del df, labeled, future_labeled, profiles, pois, test_data
        gc.collect()
    
    print(f"\nTotal time: {int(time.time() - start)}s")

if __name__ == "__main__":
    try:
        main()
    except Exception as e:
        print(f"\nERROR: {e}")
        import traceback
        traceback.print_exc()

Installing requirements...
GeoBLEU imported successfully.


CITY B
Loading: /kaggle/input/humob-data/15313913/city_B_challengedata.csv
Grid: 200 x 200

Splitting users for validation...
Train users: 21600, Val users: 5400
Building profiles for 30000 users...


Profiling:   0%|          | 0/30000 [00:00<?, ?it/s]

Profiles built: 30000
Computing top 25 POIs...
POI computation done

------------------------------------------------------------
VALIDATION (on held-out users)
------------------------------------------------------------


Val strategy_exact:   0%|          | 0/77071 [00:00<?, ?it/s]

Strategy 1 (Exact): 0.113134


...

Running spatial...


spatial:   0%|          | 0/284970 [00:00<?, ?it/s]

Running freq...


freq:   0%|          | 0/284970 [00:00<?, ?it/s]

Ensembling...
Saved: ./results/submission_D.csv (309413 predictions)

Total time: 4873s
